# UQRoute-TC — Qwen 1.5B development pilot

Run the numbered sections from top to bottom on an **L4 GPU runtime**. On a fresh Colab runtime, start again at section 1. The older reproduction notebook and its Drive results are separate; this notebook runs only five clean **development-split** cases with the new resumable runner.

The model runs through vLLM on this Colab machine (`127.0.0.1:8000`). The pilot records results in Google Drive. Wait for each cell to finish before running the next.

Use a fresh Colab runtime for this model. When changing models, reconnect or restart the runtime, then run this notebook from the top.


## 1. Restore Drive and pinned code


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount('/content/drive')
assert Path('/content/drive/MyDrive/UQRoute-TC').exists(), 'Drive folder is missing'

benchmark = Path('/content/robustbench-tc-release')
if not benchmark.exists():
    subprocess.run([
        'git', 'clone', 'https://github.com/WillChow66/robustbench-tc-release.git',
        str(benchmark),
    ], check=True)
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
actual_benchmark = subprocess.check_output(
    ['git', '-C', str(benchmark), 'rev-parse', 'HEAD'], text=True,
).strip()
assert actual_benchmark == benchmark_revision

project = Path('/content/uqroute-tc-pilot')
if not project.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'feat/task1-near-reproduction-split',
        '--single-branch', 'https://github.com/RavindraSSK/uqroute-tc.git',
        str(project),
    ], check=True)
pilot_revision = '69805b9341306692ae4a412a0a9a8939cd156504'
subprocess.run(['git', '-C', str(project), 'checkout', '--detach', pilot_revision], check=True)
assert (project / 'src/uqroute_tc/inference/pilot.py').exists()

print('Drive: ready')
print('Benchmark revision:', actual_benchmark)
print('Pilot code revision:', pilot_revision)


## 2. Install GPU packages and verify the L4


In [ ]:
import importlib.metadata
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'vllm==0.30.0', 'openai>=1.50,<3', 'tqdm>=4.66,<5',
    '-e', str(project),
], check=True)

# Colab's preinstalled TorchAudio uses a different CUDA version from vLLM's PyTorch.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=True)

import torch
import vllm

assert torch.cuda.is_available(), 'GPU is unavailable; select an L4 runtime'
gpu = torch.cuda.get_device_name(0)
assert 'L4' in gpu, f'Expected L4; got {gpu}'
print('GPU:', gpu)
print('PyTorch:', torch.__version__)
print('vLLM:', vllm.__version__)
print('Project:', importlib.metadata.version('uqroute-tc'))


## 3. Start Qwen 1.5B and wait for readiness


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import time
import urllib.request

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
model_revision = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
endpoint = 'http://127.0.0.1:8000/v1'
server_log = Path('/content/uqroute_qwen25-15b-dev-five-v1_server.log')

def served_models():
    try:
        with urllib.request.urlopen(f'{endpoint}/models', timeout=3) as response:
            return [model['id'] for model in json.load(response)['data']]
    except Exception:
        return None

print('Current runtime check:', datetime.now(timezone.utc).isoformat())
active = served_models()
if active is not None and model_id not in active:
    raise RuntimeError(f'Port 8000 is serving another model: {active}')

if active and model_id in active:
    print('Server ready:', model_id)
else:
    processes = subprocess.check_output(['ps', '-eo', 'args='], text=True)
    server = None
    if f'vllm serve {model_id}' in processes:
        print('Existing Qwen 1.5B process is starting; waiting for readiness.')
    else:
        with server_log.open('w') as log:
            server = subprocess.Popen([
                'vllm', 'serve', model_id,
                '--revision', model_revision,
                '--tokenizer-revision', model_revision,
                '--dtype', 'bfloat16',
                '--max-model-len', '8192',
                '--gpu-memory-utilization', '0.88',
                '--port', '8000',
            ], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        print('Started server PID:', server.pid)

    for _ in range(120):
        if model_id in (served_models() or []):
            break
        if server is not None and server.poll() is not None:
            print('\n'.join(server_log.read_text(errors='replace').splitlines()[-40:]))
            raise RuntimeError('Qwen 1.5B server exited during startup')
        time.sleep(5)
    else:
        if server_log.exists():
            print('\n'.join(server_log.read_text(errors='replace').splitlines()[-40:]))
        raise TimeoutError('Qwen 1.5B did not become ready')
    print('Server ready:', model_id)


## 4. Run the five development cases


In [ ]:
import importlib.metadata
import json
import subprocess
import sys
import urllib.request
from pathlib import Path

with urllib.request.urlopen(f'{endpoint}/models', timeout=5) as response:
    running = [model['id'] for model in json.load(response)['data']]
assert model_id in running, f'Qwen 1.5B is not ready: {running}'

output_dir = Path('/content/drive/MyDrive/UQRoute-TC/pilots/qwen25-15b-dev-five-v1')
command = [
    sys.executable, '-m', 'uqroute_tc.inference.pilot',
    '--benchmark-root', str(benchmark),
    '--split-manifest', str(project / 'docs/splits/group_split_v1.json'),
    '--output-dir', str(output_dir),
    '--model-id', model_id,
    '--model-revision', model_revision,
    '--tokenizer-revision', model_revision,
    '--endpoint', endpoint,
    '--engine-version', importlib.metadata.version('vllm'),
    '--dtype', 'bfloat16',
    '--max-model-len', '8192',
    '--temperature', '0.001',
    '--max-tokens', '1024',
    '--top-logprobs', '5',
    '--limit', '5',
]
subprocess.run(command, check=True)
print('Pilot saved to:', output_dir)


## 5. Score and confirm resume


In [ ]:
import json
import subprocess
import sys

subprocess.run([
    sys.executable,
    str(benchmark / 'leaderboard/scoring/eval_results.py'),
    '--predictions', str(output_dir),
    '--output', str(output_dir / 'scores.json'),
], check=True)

# A second run must skip the five completed cases and preserve one row per case.
subprocess.run(command, check=True)
rows = [json.loads(line) for line in (output_dir / 'pilot.predictions.jsonl').read_text().splitlines()]
assert len(rows) == 5 and len({row['id'] for row in rows}) == 5
assert all(row['uqroute']['status'] == 'complete' for row in rows)
assert all(row['uqroute']['evidence']['valid'] for row in rows)
print('Verified: five unique scored-ready records with valid token evidence.')
print('Saved in:', output_dir)
